# DSE 200: Day 2 Lab, Chunk 4.4
## Beyond CSV: Parquet, Arrow, DuckDB, Iceberg

**Chunk 4.4, 30 minutes, agent off for hands-on**

Same MovieLens files as Chunk 4.3. One idea: CSV is for humans, Parquet is for machines, and DuckDB lets you ask SQL questions of either without a server.

| Cell | What to do |
| --- | --- |
| `DEMO` | Already written. Run it while Emin talks. |
| `READ` | **Predict before running.** Commit to an answer. |
| `YOU TRY` | Has a `# TODO`. Yours. |
| `CHECKPOINT` | Run it. It tells you whether you are right. |

In [ ]:
# SETUP, run first. Installs two packages and finds the MovieLens files.
!uv pip install --system -q pyarrow duckdb "pyiceberg[sql-sqlite,pyarrow]"
import os, timeit, urllib.request
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import duckdb

for name in ("movies.csv", "ratings_reduced.csv"):
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)

movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings_reduced.csv")
print(f"movies {movies.shape}, ratings {ratings.shape}")
ratings.head(3)

## 4.4.1 CSV is a text file pretending to be a table

Every number in a CSV is stored as characters. `3.5` is three bytes of text that pandas has to parse into a float, every single time you read it. There is no type information, no index, and you cannot read one column without reading all of them.

Parquet stores each column separately, in its real type, compressed. Reading two columns out of twenty reads two columns.

In [ ]:
# DEMO, write the same data as Parquet, compare size and read time
ratings.to_parquet("ratings.parquet")

for f in ("ratings_reduced.csv", "ratings.parquet"):
    print(f"{f:22s} {os.path.getsize(f)/1e6:6.2f} MB")

def best_of_3(fn):                      # first call pays import and warm-up costs; take the best
    return min(timeit.repeat(fn, number=1, repeat=3))

print(f"read csv      {best_of_3(lambda: pd.read_csv('ratings_reduced.csv')):.3f}s")
print(f"read parquet  {best_of_3(lambda: pd.read_parquet('ratings.parquet')):.3f}s")
print(f"one column    {best_of_3(lambda: pd.read_parquet('ratings.parquet', columns=['rating'])):.3f}s")

In [ ]:
# READ 4.4.1, Parquet remembers dtypes, CSV does not.
# ratings['timestamp'] is an int. If we convert it to a real datetime, write to CSV, and read back,
# what dtype comes back? And from Parquet?
#
#     r = ratings.assign(when=pd.to_datetime(ratings["timestamp"], unit="s"))
#     r.to_csv("r.csv", index=False);  r.to_parquet("r.parquet")
#     pd.read_csv("r.csv")["when"].dtype,  pd.read_parquet("r.parquet")["when"].dtype
#
# Commit first.
prediction = "???"

r = ratings.assign(when=pd.to_datetime(ratings["timestamp"], unit="s"))
r.to_csv("r.csv", index=False); r.to_parquet("r.parquet")
print("your prediction:", prediction)
print("csv     ->", pd.read_csv("r.csv")["when"].dtype)
print("parquet ->", pd.read_parquet("r.parquet")["when"].dtype)

**Why it matters.** CSV threw the datetime away and handed back a string. Every CSV round trip is a chance for a column to silently change type. Parquet carries the schema with the data.

## 4.4.2 Arrow is the in-memory version of the same idea

Parquet is the file on disk. Arrow is the same columnar layout in memory. pandas, DuckDB, Polars, Spark, R, and most databases all speak Arrow, which is why you can hand a table from one to another without copying or re-parsing.

You rarely touch Arrow directly. You should know it is there, because it is the reason `pd.read_parquet` and `duckdb` are fast.

In [ ]:
# DEMO, the same table in three forms, no copying between them
table = pa.Table.from_pandas(ratings)          # pandas -> Arrow
print(table.schema)
print()
print("rows:", table.num_rows, "columns:", table.num_columns)
back = table.to_pandas()                       # Arrow -> pandas
print("round trip equal:", back.equals(ratings))

## 4.4.3 DuckDB: SQL on files, no server

DuckDB is a database that lives inside your notebook. No install beyond `pip`, no server, no connection string. It reads CSV and Parquet directly and returns pandas DataFrames.

If you already think in `groupby`, SQL is the same ideas with different spelling. `WHERE` is the filter, `GROUP BY` is the groupby, `ORDER BY ... LIMIT` is the sort.

In [ ]:
# DEMO, query the Parquet file directly
duckdb.sql("""
    SELECT rating, COUNT(*) AS n
    FROM 'ratings.parquet'
    GROUP BY rating
    ORDER BY rating
""").df()

In [ ]:
# DEMO, join a CSV to a Parquet file. DuckDB does not care which is which.
top = duckdb.sql("""
    SELECT m.title, COUNT(*) AS n, ROUND(AVG(r.rating), 2) AS avg_rating
    FROM 'ratings.parquet' r
    JOIN 'movies.csv' m USING (movieId)
    GROUP BY m.title
    HAVING n >= 100
    ORDER BY avg_rating DESC
    LIMIT 10
""").df()
top

In [ ]:
# DEMO, same question in pandas, so you can see they agree
merged = ratings.merge(movies, on="movieId")
g = merged.groupby("title")["rating"].agg(n="size", avg_rating="mean")
g = g[g["n"] >= 100].sort_values("avg_rating", ascending=False).head(10).round(2)
print("same top title:", g.index[0] == top["title"][0])
g

In [ ]:
# DEMO, DuckDB can also query a DataFrame that is already in memory, by name
duckdb.sql("SELECT COUNT(DISTINCT userId) AS users FROM ratings").df()

In [ ]:
# YOU TRY 4.4, which genre gets the most ratings?
# movies.genres looks like "Adventure|Animation|Children". One movie, many genres.
# Write one DuckDB query that returns (genre, n) for the five genres with the most ratings.
# Hint: UNNEST(STRING_SPLIT(genres, '|')) AS genre  turns one row into one row per genre.

# TODO
genre_counts = duckdb.sql("""
    SELECT ... 
""").df()
genre_counts

In [ ]:
# CHECKPOINT 4.4
expected = (ratings.merge(movies, on="movieId")
            .assign(genre=lambda d: d["genres"].str.split("|")).explode("genre")
            ["genre"].value_counts().head(5))
try:
    got = genre_counts.set_index(genre_counts.columns[0]).iloc[:, 0]
    ok = list(got.index) == list(expected.index) and list(got.values) == list(expected.values)
    print("correct" if ok else f"not yet. expected:\n{expected}")
except Exception as e:
    print("not yet:", e)


## 4.4.4 Iceberg: what this looks like at company scale

You now have `ratings.parquet`. At a company you'd have ten thousand of them: one per day, per region, per source, written by different jobs, some of them re-written when a bug was found.

Three questions appear immediately:

1. **Which files make up "the ratings table" right now?** A folder listing is not an answer once files are being added and replaced concurrently.
2. **What did the table look like last Tuesday?** Someone ran a report, the numbers changed, and they want to know why.
3. **How do I add a column without rewriting ten thousand files?**

**Apache Iceberg** is a table format: a small set of metadata files that sit next to the Parquet files and answer those three questions. Spark, Trino, Snowflake, BigQuery, and DuckDB can all read the same Iceberg table. Normally the catalog is a service. Here it's a SQLite file and the warehouse is a folder, which is enough to see the mechanics.


In [ ]:
# DEMO, an Iceberg table on your laptop: a SQLite catalog and a folder
from pyiceberg.catalog.sql import SqlCatalog
import shutil
shutil.rmtree("warehouse", ignore_errors=True); os.makedirs("warehouse")

catalog = SqlCatalog("local", uri="sqlite:///warehouse/catalog.db",
                     warehouse="file://" + os.path.abspath("warehouse"))
catalog.create_namespace("movielens")

arrow = pa.Table.from_pandas(ratings, preserve_index=False)
table = catalog.create_table("movielens.ratings", schema=arrow.schema)

table.append(arrow.slice(0, 300000))      # pretend this is Monday's load
table.append(arrow.slice(300000))         # and this is Tuesday's
print("rows now:", table.scan().to_arrow().num_rows)

In [ ]:
# DEMO, question 1 and 2: which files are the table, and what did it look like before?
for s in table.metadata.snapshots:
    print(f"snapshot {s.snapshot_id}  {s.summary.operation.value:7s} total rows {s.summary['total-records']}")

monday = table.metadata.snapshots[0].snapshot_id
print("as of Monday:", table.scan(snapshot_id=monday).to_arrow().num_rows)
print("now:         ", table.scan().to_arrow().num_rows)

print("\ndata files the current snapshot points at:")
for f in table.scan().plan_files():
    print("  ", os.path.relpath(f.file.file_path.replace("file://", "")))

In [ ]:
# DEMO, question 3: add a column. No Parquet file is rewritten.
from pyiceberg.types import StringType
before = sum(len(fs) for _, _, fs in os.walk("warehouse/movielens/ratings/data"))
with table.update_schema() as update:
    update.add_column("source", StringType())
after = sum(len(fs) for _, _, fs in os.walk("warehouse/movielens/ratings/data"))
print("columns:", [f.name for f in table.schema().fields])
print(f"data files before {before}, after {after}")

In [ ]:
# DEMO, and DuckDB reads it like anything else
current = table.scan().to_arrow()
duckdb.sql("SELECT rating, COUNT(*) AS n FROM current GROUP BY rating ORDER BY rating").df()

**What you just saw.** Two appends became two snapshots. The old snapshot is still queryable. Adding a column touched metadata only. That's the entire pitch: Parquet files plus a ledger of which files are the table and what the schema was at each point in time.

The mental model, bottom up:

| Layer | What it is | What you did today |
| --- | --- | --- |
| Arrow | columnar layout in memory | `pa.Table.from_pandas` |
| Parquet | that layout on disk, one file | `to_parquet` |
| DuckDB | engine that reads those files with SQL | `duckdb.sql` |
| Iceberg | catalog that says which files are "the table", with history | `table.append`, `scan(snapshot_id=...)` |

Nothing here replaces a database. On Day 5 you'll see when you want Postgres instead, and the answer is mostly "when many people write small changes at the same time".

In [ ]:
# DEMO, clean up the scratch files
for f in ("r.csv", "r.parquet", "ratings.parquet"):
    os.remove(f)
shutil.rmtree("warehouse")
print("done")